In [3]:
# Prep

!pip install -r "requirements ejercicio 2.txt"

import time, re
import pandas as pd
from bs4 import BeautifulSoup

# Selenium (renderiza JS)
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

BASE = "https://www.bumeran.com.pe"
START_URL = "https://www.bumeran.com.pe/en-lima/empleos-area-tecnologia-sistemas-y-telecomunicaciones-subarea-programacion-full-time-publicacion-menor-a-15-dias.html"

# ---------- Configurar navegador ----------
opts = Options()
opts.add_argument("--headless=new")       # si algo falla, prueba "--headless" o comenta esta línea
opts.add_argument("--no-sandbox")
opts.add_argument("--disable-gpu")
opts.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=opts)
wait = WebDriverWait(driver, 12)

def accept_cookies_if_present():
    """Clic en el banner de cookies si aparece (texto en ES/EN)."""
    try:
        for txt in ["Aceptar", "Acepto", "De acuerdo", "Accept", "I agree"]:
            # busca botones o enlaces con ese texto
            els = driver.find_elements(By.XPATH, f"//*[self::button or self::a][contains(., '{txt}')]")
            for el in els:
                try:
                    el.click()
                    time.sleep(0.5)
                    return
                except:
                    pass
    except:
        pass

def get_soup_js(url: str) -> BeautifulSoup:
    driver.get(url)
    accept_cookies_if_present()
    # Espera a que carguen tarjetas o el contenedor principal
    try:
        wait.until(EC.any_of(
            EC.presence_of_element_located((By.CSS_SELECTOR, "article, [data-testid='job-card'], .job-card")),
            EC.presence_of_element_located((By.CSS_SELECTOR, "main, section"))
        ))
    except:
        pass
    # Scroll por si hay carga diferida
    try:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1)
    except:
        pass
    return BeautifulSoup(driver.page_source, "html.parser")

def parse_cards(soup: BeautifulSoup):
    rows = []
    cards = soup.select("article, .list-item, .sc-card, .job-card, [data-testid='job-card']")
    for c in cards:
        a = c.select_one("a[href*='/empleo/'], a[href*='/trabajo/'], a[href*='/oferta/'], a[href*='/job/']")
        if not a:
            continue
        url = a.get("href")
        if url and url.startswith("/"):
            url = BASE + url
        title = (a.get_text(strip=True) or "").replace("\n", " ")
        company_node = c.select_one(".company, .sc-company, .card-company, [data-testid='company-name'], [class*='empresa']")
        company = company_node.get_text(strip=True) if company_node else ""
        location_node = c.select_one(".location, .sc-location, [data-testid='job-location'], [class*='ubicacion']")
        location = location_node.get_text(strip=True) if location_node else ""
        if url and title:
            rows.append({"title": title, "url": url, "company": company, "location": location})
    return rows

def build_page_url(start_url: str, page_number: int) -> str:
    # Soporta ...-p2.html o ?page=2
    if page_number == 1:
        return start_url
    if start_url.endswith(".html"):
        return start_url[:-5] + f"-p{page_number}.html"
    if "page=" in start_url:
        return re.sub(r"page=\\d+", f"page={page_number}", start_url)
    sep = "&" if "?" in start_url else "?"
    return f"{start_url}{sep}page={page_number}"

def crawl_all_pages_js(start_url: str, max_pages: int = 50) -> pd.DataFrame:
    all_rows = []
    for p in range(1, max_pages + 1):
        urlp = build_page_url(start_url, p)
        soup = get_soup_js(urlp)
        page_rows = parse_cards(soup)
        print(f"p{p}: {len(page_rows)} avisos")
        if p > 1 and len(page_rows) == 0:
            print("Sin tarjetas -> corto")
            break
        all_rows.extend(page_rows)
        time.sleep(1.0)
    return pd.DataFrame(all_rows).drop_duplicates(subset=["url"]).reset_index(drop=True)



## 2. Task Description
Scrape all Data Science job offers from the Bumeran platform that match the following filters (using code not by hand!):

In [1]:
from IPython.display import display, HTML

display(HTML(data="""
<style>
    div#notebook-container    { width: 95%; }
    div#menubar-container     { width: 65%; }
    div#maintoolbar-container { width: 99%; }a
</style>
"""))

In [43]:
from selenium import webdriver
import re
import time 
from selenium.webdriver.common.by import By
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time

In [49]:
driver = webdriver.Chrome()
url = 'https://www.bumeran.com.pe/en-lima/empleos-area-tecnologia-sistemas-y-telecomunicaciones-subarea-analisis-de-datos-publicacion-menor-a-15-dias-busqueda-data-science.html'
driver.get( url )
driver.maximize_window()

In [52]:
driver.get( url )
puesto_trabajo = driver.find_element( By.XPATH, '/html/body/div[1]/div/div[3]/div[1]/div/div/div[2]/div[1]/a' )
puesto_trabajo.click()

In [56]:
driver = webdriver.Chrome()
url_1 = "https://www.bumeran.com.pe/empleos/docente-de-data-science-modalidad-virtual-visiva-1117971443.html"
driver.get (url_1)

In [57]:
titulo = driver.find_element(By.TAG_NAME, "h1").text
titulo

'Docente de Data Science - Modalidad Virtual'

In [59]:
descripcion_puesto = driver.find_element(By.XPATH, '/html/body/div[1]/div/div/div[1]/div[3]/div/div/div/div[1]/div[1]/div[2]/div/div[1]/p').text
descripcion_puesto

'Somos VISIVA, holding educativo de CERTUS, UCAL | Universidad de Ciencias y Artes de América Latina y Toulouse Lautrec, tenemos el propósito de llenar el mundo de personas que lo mejoren.\nEn Toulouse Lautrec de Educación Continua (PEC) Lima, estamos en la búsqueda de profesionales con experiencia para dictar el siguiente el programa en modalidad virtual:\nDATA SCIENCE\nPerfil\nProfesional de la carrera de Ingeniería informática, administración, ciencia de datos, sistemas o afines.(Registrado en Minedu o Sunedu)\nContar con dominio en el uso y metodologías de análisis de datos, dominio de Python, SQL y librerías de análisis/visualización.\nCapacidad para guiar proyectos prácticos e integradores en entornos educativos.\nContar con mínimo de 3 años de experiencia en el análisis de datos.\nDisponibilidad Virtual: Lunes y Miércoles y/o Martes y Jueves de (8:00pm a 10:30pm)\nVISIVA mantiene una política de contratación inclusiva por lo que invita a todas las personas a participar en el pro

In [69]:
distrito = driver.find_element(By.XPATH, "/html/body/div[1]/div/div/div[1]/div[3]/div/div/div/div[1]/div[1]/div[2]/div/div[1]/div[1]/div[2]/div/div/li/a/h2").text
distrito

'Lima, Lima, Peru'

In [71]:
modalidad = driver.find_element(By.XPATH, "/html/body/div[1]/div/div/div[1]/div[3]/div/div/div/div[1]/div[1]/div[2]/div/div[1]/div[4]/div/ul/div[1]/li[1]/a/p").text
modalidad

'Remoto'

In [89]:
import csv

nombre_archivo = "trabajo.csv"

with open(nombre_archivo, mode="w", newline="", encoding="utf-8-sig") as file:
    writer = csv.writer(file, delimiter=";")  # Excel usa ; como separador en español
    writer.writerow(["Job Title", "Description", "District", "Work Mode"])
    writer.writerow([titulo, descripcion_puesto, distrito, modalidad])

print(f"Archivo CSV generado: {nombre_archivo}")

Archivo CSV generado: trabajo.csv


In [93]:
import pandas as pd
df = pd.read_csv("trabajo.csv", encoding="utf-8-sig", delimiter=";")
df

,Job Title,Description,District,Work Mode
0,Docente de Data Science - Modalidad Virtual,"Somos VISIVA, holding educativo de CERTUS, UCA...","Lima, Lima, Peru",Remoto
